# Pharmacy Medicine Demand & Expiry-Risk Forecasting

Run this notebook in Google Colab to regenerate the included synthetic data, train the demand model, evaluate a 12-week holdout, forecast future demand, and estimate FEFO expiry exposure. All data is fictional; no patient or pharmacy records are used.

In [1]:
from pathlib import Path
import os
import subprocess
import sys

REPO_DIR = Path('/content/Assignments')
PROJECT_DIR = REPO_DIR / 'pharmacy-medicine-demand-expiry-risk-forecasting'
if not PROJECT_DIR.exists():
    if not REPO_DIR.exists():
        subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/nithinkumarreddy45/Assignments.git',
            str(REPO_DIR)
        ], check=True)
    if not PROJECT_DIR.exists():
        raise FileNotFoundError(f'Project folder not found: {PROJECT_DIR}')

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print('Project folder:', PROJECT_DIR)

Project folder: /content/Assignments/pharmacy-medicine-demand-expiry-risk-forecasting


## Generate data and run the forecasting pipeline

The pipeline creates 24 fictional SKUs with 104 weeks of generated demand, performs a chronological holdout evaluation, refits the model on all history, and writes forecast and expiry-risk evidence to the project folders.

In [2]:
from src.generate_data import build_dataset
from src.pipeline import run_pipeline
import pandas as pd

build_dataset()
summary = run_pipeline()
pd.Series(summary, name='value').to_frame()

,value
n_skus,24
history_weeks_per_sku,104
backtest_horizon_weeks,12
forecast_horizon_weeks,12
aggregate_wape,0.134803
naive_4_week_average_wape,0.137468
wape_change_vs_naive_pct,1.938294
mean_sku_mae,6.339962
snapshot_date,2026-10-26


## Inspect evaluation and forecast output

In [3]:
metrics = pd.read_csv(PROJECT_DIR / 'evidence/holdout_metrics_by_sku.csv')
forecasts = pd.read_csv(PROJECT_DIR / 'evidence/12_week_demand_forecast.csv')
expiry = pd.read_csv(PROJECT_DIR / 'evidence/expiry_risk_and_reorder.csv')

display(metrics.describe().round(3))
display(forecasts.head(12))
display(expiry.sort_values('risk_value_estimate', ascending=False).head(10))

,mae,wape,naive_mae,naive_wape
count,24.000,24.000,24.000,24.000
mean,6.340,0.147,6.465,0.152
std,1.895,0.045,1.500,0.045
min,3.692,0.075,3.875,0.084
25%,5.101,0.120,5.479,0.119
50%,6.141,0.140,6.229,0.148
75%,7.280,0.183,7.188,0.183
max,11.212,0.232,9.750,0.247


,week_start,sku_id,forecast_units,approx_p10_units,approx_p90_units
0,2026-10-26,MED001,88.4,78.0,98.9
1,2026-11-02,MED001,87.5,77.0,97.9
2,2026-11-09,MED001,88.4,77.9,98.8
3,2026-11-16,MED001,88.0,77.6,98.5
4,2026-11-23,MED001,90.2,79.8,100.7
5,2026-11-30,MED001,89.9,79.4,100.3
6,2026-12-07,MED001,90.8,80.4,101.3
7,2026-12-14,MED001,90.7,80.2,101.1
8,2026-12-21,MED001,91.5,81.1,102.0
9,2026-12-28,MED001,91.6,81.2,102.1


,sku_id,starting_on_hand_units,forecast_demand_12w,units_expiring_unconsumed_12w,expiry_risk_pct_of_starting_stock,risk_value_estimate,recommended_order_units,category,unit_cost,lead_time_weeks
16,MED017,474,688.1,52,11.0,358.8,0,Chronic Care,6.90,1
0,MED001,677,1081.1,0,0.0,0.0,0,Analgesic,8.31,3
2,MED003,486,963.0,0,0.0,0.0,0,Analgesic,3.55,3
1,MED002,358,500.3,0,0.0,0.0,0,Analgesic,3.60,3
4,MED005,432,849.2,0,0.0,0.0,0,Analgesic,14.05,1
5,MED006,401,593.9,0,0.0,0.0,0,Analgesic,7.03,1
6,MED007,168,302.8,0,0.0,0.0,0,Analgesic,13.74,3
3,MED004,230,482.5,0,0.0,0.0,0,Analgesic,6.72,2
7,MED008,759,842.6,0,0.0,0.0,0,Analgesic,9.61,3
8,MED009,251,745.1,0,0.0,0.0,0,Cold and Flu,10.48,1


## How to read the results

The backtest compares recursive 12-week forecasts against a baseline that repeats the recent four-week average. The included synthetic run showed only a modest WAPE improvement; this is not evidence of real-world performance. Expiry-risk estimates simulate first-expiry-first-out consumption and assume no replenishment during the forecast horizon. Treat the suggested reorder quantity as an educational estimate, not an automatic purchasing instruction.

AI assistance was used to draft parts of the project code and documentation. Review and understand the implementation before submitting it.